# Week 9 Fed-LoRA comparison on Qwen2.5-1.5B

This notebook downloads the pinned VASTLoRA checkout and runs the `qwen15b-fedlora` profile on Kaggle T4 x2. The profile compares raw, freshness, FedAvg-LoRA, FedEx-LoRA, FLoRA, FFA-LoRA, FLoRG and FedRot across two regimes and three seeds.

Set `MODE='smoke'` first. Then use `development` and only run `confirmation` after the development protocol is frozen. Existing valid `result.json` files are skipped; `RESUME_ROOTS` can import valid results from an attached Kaggle dataset.

In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import time
import zipfile
from IPython.display import display, FileLink

REPO_URL = 'https://github.com/TrgPhan/VASTLoRA.git'
REPO_REF = 'main'  # replace with the printed commit after pushing
MODE = 'preflight'  # preflight / smoke / development / confirmation
RUN_TRAINING = False
CONFIRM_PROTOCOL_FROZEN = False
GPU_IDS = [0, 1]
WORK_ROOT = Path('/kaggle/working')
OUTPUT_ROOT = WORK_ROOT / f'week9_1_5b_fedlora_{MODE}'
RESUME_ROOTS = []  # e.g. ['/kaggle/input/old-fedlora-output/week9_1_5b_fedlora_development']
MAX_JOBS = None  # limits only newly pending jobs
RETRY_INCOMPLETE = False
RUN_ANALYSIS = False

if MODE not in {'preflight', 'smoke', 'development', 'confirmation'}:
    raise ValueError('invalid MODE')
if MODE == 'preflight' and RUN_TRAINING:
    raise ValueError('preflight never trains; choose smoke, development or confirmation')
if MODE == 'confirmation' and RUN_TRAINING and not CONFIRM_PROTOCOL_FROZEN:
    raise RuntimeError('Set CONFIRM_PROTOCOL_FROZEN=True only after freezing development settings.')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({'mode': MODE, 'training': RUN_TRAINING, 'output_root': str(OUTPUT_ROOT)})

In [ ]:
REPO_DIR = WORK_ROOT / 'VASTLoRA-fedlora'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR, check=True)
if REPO_REF != 'main':
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO_DIR, check=True)
resolved = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True).strip()
if dirty:
    raise RuntimeError('The Kaggle checkout is dirty.')
RUNNER = REPO_DIR / 'scripts/run_week9_generation.py'
if not RUNNER.exists():
    raise RuntimeError('The Week 9 runner is missing from the checkout.')
print({'repo': str(REPO_DIR), 'commit': resolved})

In [ ]:
# Keep Kaggle's CUDA torch build and install the tested runtime.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
constraints = WORK_ROOT / 'week9_torch_constraint.txt'
import importlib.metadata as metadata
constraints.write_text('torch==' + metadata.version('torch') + '\n', encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), '-e', '.[scale,dev,generation]',
                'transformers==5.10.2', 'peft==0.20.0', 'accelerate==1.14.0',
                'bitsandbytes==0.50.2', 'datasets==5.0.0', 'rouge-score==0.1.2'],
               cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-c',
    'import torch, peft, transformers, bitsandbytes; print({\"torch\":torch.__version__, \"peft\":peft.__version__, \"transformers\":transformers.__version__, \"bitsandbytes\":bitsandbytes.__version__})'
], check=True)

In [ ]:
profile = 'qwen15b-fedlora'
base = [sys.executable, '-u', str(RUNNER), '--profile', profile, '--output-root', str(OUTPUT_ROOT)]
if MODE == 'smoke':
    base += ['--smoke']
else:
    base += ['--phase', 'development' if MODE == 'preflight' else MODE]

def with_resume(command):
    command = list(command)
    for root in RESUME_ROOTS:
        command += ['--resume-root', str(root)]
    return command

preflight = base + ['--dry-run']
subprocess.run(preflight, cwd=REPO_DIR, check=True)
subprocess.run(base + ['--prepare-only'], cwd=REPO_DIR, check=True)
subprocess.run(with_resume(base + ['--plan-only']), cwd=REPO_DIR, check=True)
plan = json.loads((OUTPUT_ROOT / 'job_plan.json').read_text())
matrix = json.loads((OUTPUT_ROOT / 'matrix.json').read_text())
display({'commit': resolved, 'jobs': len(plan), 'methods': matrix['methods'],
         'reusable': sum(item['status'] in {'complete', 'importable'} for item in plan),
         'pending': sum(item['status'] == 'pending' for item in plan),
         'output_root': str(OUTPUT_ROOT)})

In [ ]:
if RUN_TRAINING:
    import torch
    if not torch.cuda.is_available() or len(GPU_IDS) != 2:
        raise RuntimeError(f'Expected two CUDA devices, found {torch.cuda.device_count()}')
    gpu_info = subprocess.check_output(['nvidia-smi'], text=True)
    print(gpu_info)
    first_config = json.loads(Path(plan[0]['command'][plan[0]['command'].index('--config') + 1]).read_text())
    from huggingface_hub import snapshot_download
    snapshot_download(first_config['model']['name'], revision=first_config['model']['revision'])
    command = with_resume(base)
    for gpu in GPU_IDS:
        command += ['--gpu', str(gpu)]
    if MAX_JOBS is not None:
        command += ['--max-jobs', str(MAX_JOBS)]
    if RETRY_INCOMPLETE:
        command += ['--retry-incomplete']
    log_path = OUTPUT_ROOT / 'launcher.log'
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print({'pid': process.pid, 'log': str(log_path), 'command': command})
        while process.poll() is None:
            print(log_path.read_text(encoding='utf-8', errors='replace')[-3000:], flush=True)
            time.sleep(30)
    if process.returncode:
        raise RuntimeError(f'launcher failed with code {process.returncode}; inspect {log_path}')
else:
    print('Preflight only: no training jobs were started.')

In [ ]:
if RUN_ANALYSIS:
    subprocess.run([sys.executable, str(REPO_DIR / 'scripts/analyze_week9_generation.py'),
                    '--input-dir', str(OUTPUT_ROOT), '--target', 'flora_lora'],
                   cwd=REPO_DIR, check=True)
archive = WORK_ROOT / (OUTPUT_ROOT.name + '.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file() and path.name != '.launcher.lock':
            z.write(path, path.relative_to(OUTPUT_ROOT.parent))
display(FileLink(str(archive)))